# Unified TPU Training - Qwen3-1.7B (parameterized: 1k vs full)

Single notebook that replaces `[TPU_Training]...1k_len` and `...full_len`. The 1k-vs-full difference (gene truncation, `GRAD_ACCUM_STEPS`, repo/dirs) is now controlled by the **parameters** cell below and can be set by the Colab frontend (`colab_frontends/run_train_colab.ipynb`) via papermill.

In [ ]:
# ===== Papermill parameters (defaults reproduce the "1k_len" training variant) =====
# The frontend overrides these. The HF token is read from os.environ["HF_TOKEN"] (never hardcoded).

# Repo / data
REPO_URL = "https://github.com/HangYu8123/SC_Ageing_Prediction.git"
PROJECT_DIR = "/content/SC_Ageing_Prediction"
DATA_SUBDIR = "fine_tune_chunks"
REFRESH_PROJECT = False          # keep False so a frontend-supplied clone is reused

# Model / checkpoint
MODEL_ID = "Ha-ya/QWEN3-1.7B-EXTENDED"
HF_REPO_ID = "DaisyCuttie/QWEN3-1.7B-EIGHT-ORGANS-EXTENDED-1K"   # full: "...-FULL-LENGTH"
OUTPUT_DIR = "/content/drive/MyDrive/qwen3_1k_tpu_out"           # full: "...qwen3_full_tpu_out"
MERGED_DIR = "/content/drive/MyDrive/merged_model_1k"            # full: "...merged_model_full"

# Gene-text handling (THE 1k-vs-full difference)
TRUNCATE_GENES = True            # True = 1k (keep top MAX_GENES genes); full: False
MAX_GENES = 1000

# Split
TRAIN_PARTS = list(range(1, 11))
EVAL_PART = 11

# Training hyperparameters
MAX_LENGTH = 1024
NUM_EPOCHS = 3
LR = 5e-5
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
GLOBAL_TRAIN_BS = 16
GLOBAL_EVAL_BS = 128
GRAD_ACCUM_STEPS = 2             # full: 1
SEED = 42

# Upload / runtime side-effects (safe defaults for papermill)
PUSH_TO_HUB = False              # True to merge the LoRA adapter and upload to HF_REPO_ID
CLEAN_REMOTE = True              # when pushing, wipe the HF repo first
RELEASE_RUNTIME = False          # never True under papermill (it kills the kernel)


## Session 1 - Runtime and Project Setup

Steps:
1. Mount Google Drive and prepare the Colab workspace.
2. Clone the training project into `/content`.
3. Install TPU/XLA-compatible dependencies and verify the runtime.


In [ ]:
# Project setup (papermill-safe). Drive is mounted by the frontend; mount here only if needed.
import os, shutil, subprocess

if not os.path.isdir("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped:", repr(exc))

if os.path.isdir("/content"):
    os.chdir("/content")

if REFRESH_PROJECT and os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
if not os.path.exists(PROJECT_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, PROJECT_DIR])

# HF token comes from the environment (set by the frontend); never hardcoded.
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token detected in environment.")
else:
    print("No HF_TOKEN in environment (only required when PUSH_TO_HUB=True).")

print("Project directory:", PROJECT_DIR)


In [ ]:
import os
import subprocess
import sys

os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_BF16"] = "1"
os.environ.setdefault("PT_XLA_DEBUG", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PYTORCH_VERSION = "2.6.0"

def pip_install(packages: list[str], extra_args: list[str] | None = None) -> None:
    args = [sys.executable, "-m", "pip", "install"]
    if extra_args:
        args.extend(extra_args)
    args.extend(packages)
    print("Running:", " ".join(args))
    subprocess.check_call(args)

try:
    import torch  # noqa: F401
    import torch_xla  # noqa: F401
    print("torch:", torch.__version__)
    print("torch_xla:", torch_xla.__version__)
except Exception as exc:
    print("Installing torch_xla because import failed:", repr(exc))
    pip_install(
        [
            f"torch=={PYTORCH_VERSION}",
            f"torch_xla[tpu]=={PYTORCH_VERSION}",
        ],
        extra_args=["-q", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
    )

pip_install(
    [
        "transformers>=4.45.0",
        "datasets>=2.19.0",
        "accelerate>=0.33.0",
        "peft>=0.12.0",
        "huggingface_hub>=0.24.0",
        "evaluate>=0.4.2",
        "scikit-learn>=1.3.0",
    ],
    extra_args=["-q", "-U"],
)

print("Dependency setup complete. Restart the runtime if Colab asks for it.")


In [ ]:
import torch
import torch.utils.checkpoint
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
import torch_xla.utils.checkpoint as xla_ckpt

xr.use_spmd()

if not hasattr(torch, "xla"):
    torch.xla = torch_xla

torch.utils.checkpoint.checkpoint = xla_ckpt.checkpoint

print("torch:", torch.__version__)
print("torch_xla:", torch_xla.__version__)
print("PJRT device:", os.environ["PJRT_DEVICE"])
print("Global runtime device count:", xr.global_runtime_device_count())
print("Supported XLA devices:", xm.get_xla_supported_devices())
print("Current XLA device:", torch_xla.device())


## Session 2 - Experiment Configuration

Steps:
1. Define paths, model IDs, and training split choices.
2. Set training hyperparameters and numeric precision.
3. Seed the run for reproducibility.


In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import torch

DATA_DIR = Path(PROJECT_DIR) / DATA_SUBDIR
DTYPE = torch.bfloat16

ORGAN_PREFIXES = {
    "bladder": "bladder",
    "brain": "brain",
    "bone": "bone-marrow",
    "limb": "limb-muscle",
    "kidney": "kidney",
    "liver": "liver",
    "lung": "lung",
    "heart": "heart",
}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

print("Data directory:", DATA_DIR)
print("Model ID:", MODEL_ID)
print("Train parts:", TRAIN_PARTS, "| Eval part:", EVAL_PART)
print("MAX_LENGTH:", MAX_LENGTH, "| TRUNCATE_GENES:", TRUNCATE_GENES,
      "| GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("DTYPE:", DTYPE)


## Session 3 - Dataset Loading

Steps:
1. Build the train and validation file lists for all organs.
2. Load the JSON files into Hugging Face Datasets.
3. Sanity-check the dataset schema and label field.


In [ ]:
import glob
import json

from datasets import Dataset, DatasetDict, load_dataset

print("Files found:", len(glob.glob(f"{DATA_DIR}/*.json")))
print("Example files:", sorted(glob.glob(f"{DATA_DIR}/*.json"))[:5])

def build_split_files(parts: list[int]) -> list[str]:
    files = []
    for organ_prefix in ORGAN_PREFIXES.values():
        for part in parts:
            files.append(str(DATA_DIR / f"{organ_prefix}_cell_data_part_{part}.json"))
    return files

def load_json_dataset(files: list[str]) -> Dataset:
    try:
        return load_dataset("json", data_files=files, split="train")
    except Exception as exc:
        print("Falling back to manual JSON loading:", exc)
        rows = []
        for file_path in files:
            with open(file_path, "r") as handle:
                rows.extend(json.load(handle))
        return Dataset.from_list(rows)

train_files = build_split_files(TRAIN_PARTS)
eval_files = build_split_files([EVAL_PART])

print("Train file sample:", train_files[:2], "...", train_files[-1])
print("Eval file sample:", eval_files[0])

train_ds = load_json_dataset(train_files)
eval_ds = load_json_dataset(eval_files)

raw = DatasetDict({"train": train_ds, "validation": eval_ds})
print(raw)
print("Columns:", raw["train"].column_names)
print("Example output label:", raw["train"][0].get("output"))

## Session 4 - Text and Label Preparation

Steps:
1. Map age labels to integer class IDs.
2. Convert each example into a classification prompt.
3. Build the final train and validation text dataset.


In [ ]:
LABEL2ID = {"1m": 0, "3m": 1, "18m": 2, "24m": 3, "30m": 4}
ID2LABEL = {label_id: label for label, label_id in LABEL2ID.items()}
LABEL_SUFFIX = "Answer with exactly one label from {1m, 3m, 18m, 24m, 30m}."

# Map literal string numbers to actual integers for sorting
NUM_MAP = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20
}

def sort_and_truncate_genes(cell_input: str, max_genes: int = 1000) -> str:
    if not cell_input.startswith("Genes: "):
        return cell_input

    parts = cell_input.split("\n")
    genes_line = parts[0][len("Genes: "):]

    tokens = genes_line.split()
    gene_pairs = []
    for i in range(0, len(tokens) - 1, 2):
        gene = tokens[i]
        count_str = tokens[i+1]
        gene_pairs.append((gene, count_str))

    # Sort descending by mapped numerical value
    gene_pairs.sort(key=lambda pair: NUM_MAP.get(pair[1].lower(), 0), reverse=True)

    # Truncate to top max_genes
    gene_pairs = gene_pairs[:max_genes]

    # Reconstruct the line
    new_genes_line = "Genes: " + " ".join([f"{g} {c}" for g, c in gene_pairs])
    parts[0] = new_genes_line

    return "\n".join(parts)

def format_text(example: dict) -> dict:
    instruction = (example.get("instruction") or "").strip()
    cell_input = (example.get("input") or "").strip()
    label = (example.get("output") or "").strip()

    if label not in LABEL2ID:
        raise ValueError(f"Unexpected label: {label}")

    # Process the input to sort and truncate genes
    processed_input = (
        sort_and_truncate_genes(cell_input, max_genes=MAX_GENES)
        if TRUNCATE_GENES else cell_input
    )

    text = f"{instruction}\n\n{processed_input}\n\n{LABEL_SUFFIX}"
    return {"text": text, "label": LABEL2ID[label]}

raw = raw.map(format_text, remove_columns=raw["train"].column_names)
print(raw["train"][0])


## Session 5 - Tokenizer, Model, and Trainer Setup

Steps:
1. Tokenize the text dataset with fixed-length padding for TPU stability.
2. Load the sequence classification model and attach LoRA adapters.
3. Build the Trainer, metrics, and TPU sync callbacks.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.bos_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_batch(batch: dict) -> dict:
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    encoded["labels"] = batch["label"]
    return encoded

tokenized = raw.map(tokenize_batch, batched=True, remove_columns=raw["train"].column_names)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=128)

SMALL_EVAL_SIZE = 512
small_eval_dataset = tokenized["validation"].shuffle(seed=SEED).select(
    range(min(SMALL_EVAL_SIZE, len(tokenized["validation"])))
)

print(tokenized)
print("Small eval size:", len(small_eval_dataset))


In [ ]:
import inspect

import torch
import torch_xla
import torch_xla.runtime as xr
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from transformers.trainer_utils import EvalPrediction

xr.use_spmd()
num_devices = xr.global_runtime_device_count()

if GLOBAL_TRAIN_BS % num_devices != 0:
    raise ValueError(
        f"GLOBAL_TRAIN_BS ({GLOBAL_TRAIN_BS}) must be divisible by num_devices ({num_devices})."
    )
if GLOBAL_EVAL_BS % num_devices != 0:
    raise ValueError(
        f"GLOBAL_EVAL_BS ({GLOBAL_EVAL_BS}) must be divisible by num_devices ({num_devices})."
    )

def guess_lora_targets(model: torch.nn.Module) -> list[str]:
    common_names = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    present = set()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            for target_name in common_names:
                if name.endswith(target_name):
                    present.add(target_name)
    if not present:
        present = {"q_proj", "v_proj"}
    return sorted(present | {"embed_tokens"})

def filter_kwargs_for_callable(fn, kwargs: dict) -> dict:
    valid_names = set(inspect.signature(fn).parameters.keys())
    return {key: value for key, value in kwargs.items() if key in valid_names}

def compute_metrics(prediction: EvalPrediction) -> dict:
    logits = prediction.predictions
    labels = prediction.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

model_kwargs = {
    "num_labels": len(LABEL2ID),
    "id2label": ID2LABEL,
    "label2id": LABEL2ID,
    "trust_remote_code": True,
}

pretrained_signature = inspect.signature(AutoModelForSequenceClassification.from_pretrained)
if "dtype" in pretrained_signature.parameters:
    model_kwargs["dtype"] = DTYPE
else:
    model_kwargs["torch_dtype"] = DTYPE

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, **model_kwargs)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
if getattr(model.config, "bos_token_id", None) is None and tokenizer.bos_token_id is not None:
    model.config.bos_token_id = tokenizer.bos_token_id
if hasattr(model, "generation_config") and getattr(model.generation_config, "bos_token_id", None) is None:
    model.generation_config.bos_token_id = tokenizer.bos_token_id

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=guess_lora_targets(model),
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": True}
        )
    except TypeError:
        model.gradient_checkpointing_enable()

train_dataset_size = len(tokenized["train"])
effective_batch_size = GLOBAL_TRAIN_BS * GRAD_ACCUM_STEPS
steps_per_epoch = max(1, train_dataset_size // effective_batch_size)
steps_per_mid_eval = max(1, steps_per_epoch // 8)

class MidEvalCallback(TrainerCallback):
    def __init__(self, eval_dataset, every_n_steps: int):
        self.eval_dataset = eval_dataset
        self.every_n_steps = every_n_steps
        self.trainer = None

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step <= 0 or state.global_step % self.every_n_steps != 0:
            return control
        if getattr(args, "eval_steps", None) and args.eval_steps > 0 and state.global_step % args.eval_steps == 0:
            return control
        if getattr(args, "save_steps", None) and args.save_steps > 0 and state.global_step % args.save_steps == 0:
            return control

        metrics = self.trainer.evaluate(
            eval_dataset=self.eval_dataset,
            metric_key_prefix="mid_eval",
        )
        self.trainer.log(metrics)
        return control

class XlaSyncEverySubstepCallback(TrainerCallback):
    def on_substep_end(self, args, state, control, **kwargs):
        torch_xla.sync()
        return control

class PrintTrainLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if "loss" in logs:
            lr = logs.get("learning_rate")
            if lr is None:
                print(f"[train_loss] step={state.global_step} loss={logs['loss']}", flush=True)
            else:
                print(
                    f"[train_loss] step={state.global_step} loss={logs['loss']} lr={lr}",
                    flush=True,
                )
        return control

training_args_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": NUM_EPOCHS,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "optim": "adafactor",
    "per_device_train_batch_size": GLOBAL_TRAIN_BS,
    "per_device_eval_batch_size": GLOBAL_EVAL_BS,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "bf16": True,
    "logging_steps": 512,
    "save_strategy": "steps",
    "save_steps": 1000,
    "eval_strategy": "steps",
    "eval_steps": 10000,
    "dataloader_drop_last": True,
    "dataloader_pin_memory": False,
    "dataloader_num_workers": 0,
    "save_safetensors": False,
    "remove_unused_columns": False,
    "report_to": "none",
    "seed": SEED,
}

training_args_signature = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
if (
    "eval_strategy" in training_args_kwargs
    and "eval_strategy" not in training_args_signature
    and "evaluation_strategy" in training_args_signature
):
    training_args_kwargs["evaluation_strategy"] = training_args_kwargs.pop("eval_strategy")

training_args_kwargs = filter_kwargs_for_callable(TrainingArguments.__init__, training_args_kwargs)
training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized["train"],
    "eval_dataset": tokenized["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_init_signature = set(inspect.signature(Trainer.__init__).parameters.keys())
if "processing_class" in trainer_init_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_init_signature:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
trainer.add_callback(XlaSyncEverySubstepCallback())
trainer.add_callback(PrintTrainLossCallback())

mid_eval_callback = MidEvalCallback(eval_dataset=small_eval_dataset, every_n_steps=steps_per_mid_eval)
mid_eval_callback.trainer = trainer
trainer.add_callback(mid_eval_callback)

print("Trainer ready.")
print("Train samples:", train_dataset_size)
print("Effective batch size:", effective_batch_size)
print("Steps per epoch:", steps_per_epoch)
print("Mid-eval interval:", steps_per_mid_eval)


## Session 6 - Training and Adapter Export

Steps:
1. Launch TPU fine-tuning with the Trainer.
2. Save the LoRA adapter checkpoint.
3. Save the tokenizer alongside the adapter output.


In [ ]:
import os
import traceback
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None
if os.path.exists(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

try:
    if last_checkpoint is not None:
        print(f"Resuming training from checkpoint: {last_checkpoint}")
        train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print("No checkpoint found. Starting training from scratch.")
        train_result = trainer.train()

    print(train_result)

    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print("Saved adapter output to:", OUTPUT_DIR)

    # Send success email
    try:
        send_email_notification("Colab Training Completed!", f"Your training session finished successfully.\nModel saved to: {OUTPUT_DIR}")
    except NameError:
        print("Success email could not be sent because 'send_email_notification' is not defined. Please run the email cell first.")

except Exception as e:
    error_msg = traceback.format_exc()
    print("Training failed with error:\n", error_msg)

    # Send failure email
    try:
        send_email_notification("Colab Training Failed", f"Your training session encountered an error:\n\n{error_msg}")
    except NameError:
        print("Failure email could not be sent because 'send_email_notification' is not defined. Please run the email cell first.")

    raise e

## Session 7 - Adapter Merge and Hugging Face Upload

> Add blockquote



Steps:
1. Log in to Hugging Face with `HF_TOKEN`.
2. Merge the LoRA adapter back into the base model.
3. Save, validate, and upload the merged model artifacts.

Notes:
- This session expects `HF_TOKEN` to be set in the environment.
- Run it only after Session 6 has finished successfully.


In [ ]:
import os
import shutil
from datetime import datetime

import torch
from huggingface_hub import HfApi, create_repo, login
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# HF token comes from the environment (set by the frontend); never hardcoded.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not PUSH_TO_HUB:
    print("PUSH_TO_HUB=False -> skipping LoRA merge and Hugging Face upload.")
elif not HF_TOKEN:
    raise ValueError("PUSH_TO_HUB=True but no HF_TOKEN found in the environment.")
elif not os.path.exists(OUTPUT_DIR):
    raise FileNotFoundError(f"Adapter output directory not found: {OUTPUT_DIR}")
else:
    login(token=HF_TOKEN)

    if os.path.exists(MERGED_DIR):
        shutil.rmtree(MERGED_DIR)
    os.makedirs(MERGED_DIR, exist_ok=True)

    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABEL2ID),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

    model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    model = model.merge_and_unload()
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id

    embedding_rows = model.get_input_embeddings().weight.shape[0]
    tokenizer_size = len(tokenizer)

    if tokenizer_size > embedding_rows:
        model.resize_token_embeddings(tokenizer_size)
        embedding_rows = model.get_input_embeddings().weight.shape[0]

    model.config.vocab_size = embedding_rows
    if hasattr(model.config, "text_config") and hasattr(model.config.text_config, "vocab_size"):
        model.config.text_config.vocab_size = embedding_rows

    model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)

    del model
    del base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    merged_model = AutoModelForSequenceClassification.from_pretrained(MERGED_DIR, trust_remote_code=True)
    merged_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)

    assert merged_model.get_input_embeddings().weight.shape[0] == merged_model.config.vocab_size

    create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)

    api = HfApi()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if CLEAN_REMOTE:
        existing_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type="model")
        for path_in_repo in existing_files:
            if path_in_repo == ".gitattributes":
                continue
            api.delete_file(
                repo_id=HF_REPO_ID,
                repo_type="model",
                path_in_repo=path_in_repo,
                commit_message=f"Clean repo before upload - {timestamp}",
            )

    api.upload_folder(
        folder_path=MERGED_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message=f"Upload merged model and tokenizer - {timestamp}",
    )

    print("Merged model validated and uploaded to:", HF_REPO_ID)
    print("Merged tokenizer size:", len(merged_tokenizer))


## Session 8 - Optional Colab Cleanup

Steps:
1. Release the runtime after training and upload are complete.


In [ ]:
# Optional Colab cleanup. Disabled by default; under papermill this would kill the kernel.
if RELEASE_RUNTIME:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print("runtime.unassign skipped:", repr(exc))
else:
    print("RELEASE_RUNTIME=False -> keeping runtime alive.")
